# Project_3

Dataset(s) to be used: [https://data.cityofnewyork.us/City-Government/Citywide-Payroll-Data-Fiscal-Year-/k397-673e/about_data]

Analysis question: [Is there a significant difference in base salary among employees working in different boroughs of New York City?]

Columns that will (likely) be used:

[work_location_borough]

[base_salary]

Hypothesis: [Employees working in Manhattan have significantly higher average base salaries than those in other boroughs, because Manhattan is the commercial and financial center of NYC with higher living costs.]

In [47]:
import pandas as pd
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import HTML

In [48]:
df = pd.read_csv('https://data.cityofnewyork.us/resource/k397-673e.csv')
df.head()

,fiscal_year,payroll_number,agency_name,last_name,first_name,mid_init,agency_start_date,work_location_borough,title_description,leave_status_as_of_june_30,base_salary,pay_basis,regular_hours,regular_gross_paid,ot_hours,total_ot_paid,total_other_pay
0,2025,67,ADMIN FOR CHILDREN'S SVCS,WILLIAMS,JAMAL,D,2015-11-30T00:00:00.000,MANHATTAN,DIRECTOR OF FIELD OPERATIONS,ACTIVE,128299.0,per Annum,1820.0,125840.54,0.00,0.00,3000.00
1,2025,67,ADMIN FOR CHILDREN'S SVCS,SELBY,KEON,B,2022-08-01T00:00:00.000,BROOKLYN,SPECIAL OFFICER,CEASED,39322.0,per Annum,0.0,0.00,0.00,0.00,4460.24
2,2025,67,ADMIN FOR CHILDREN'S SVCS,SHEBIOBA,SHAUNETTE,A,2007-07-16T00:00:00.000,QUEENS,CHILD PROTECTIVE SPECIALIST SUPERVISOR,ACTIVE,100101.0,per Annum,1820.0,96985.85,255.75,17652.91,9936.77
3,2025,67,ADMIN FOR CHILDREN'S SVCS,OLIVERO,KENTNIA,R,2008-08-11T00:00:00.000,MANHATTAN,CHILD WELFARE SPECIALIST SUPERVISOR,ACTIVE,101758.0,per Annum,1820.0,98590.49,113.50,7469.35,8195.62
4,2025,67,ADMIN FOR CHILDREN'S SVCS,PUROHIT,PRASHANT,S,2022-10-10T00:00:00.000,MANHATTAN,STAFF ANALYST,ACTIVE,69702.0,per Annum,1820.0,69031.86,0.00,0.00,2341.29


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   fiscal_year                 1000 non-null   int64  
 1   payroll_number              1000 non-null   int64  
 2   agency_name                 1000 non-null   object 
 3   last_name                   1000 non-null   object 
 4   first_name                  1000 non-null   object 
 5   mid_init                    722 non-null    object 
 6   agency_start_date           1000 non-null   object 
 7   work_location_borough       1000 non-null   object 
 8   title_description           1000 non-null   object 
 9   leave_status_as_of_june_30  1000 non-null   object 
 10  base_salary                 1000 non-null   float64
 11  pay_basis                   1000 non-null   object 
 12  regular_hours               1000 non-null   float64
 13  regular_gross_paid          1000 n

In [50]:
print("\nWork location borough distribution:")
borough_counts = df['work_location_borough'].value_counts()
print(borough_counts)


Work location borough distribution:
work_location_borough
MANHATTAN    514
BROOKLYN     180
BRONX        180
QUEENS        95
RICHMOND      31
Name: count, dtype: int64


In [51]:
print("\nBasic statistics for base salary:")
print(f"Average base salary: ${df['base_salary'].mean():,.2f}")
print(f"Median base salary: ${df['base_salary'].median():,.2f}")
print(f"Standard deviation: ${df['base_salary'].std():,.2f}")
print(f"Minimum base salary: ${df['base_salary'].min():,.2f}")
print(f"Maximum base salary: ${df['base_salary'].max():,.2f}")


Basic statistics for base salary:
Average base salary: $84,705.33
Median base salary: $70,156.00
Standard deviation: $32,937.49
Minimum base salary: $37.67
Maximum base salary: $273,352.00


In [52]:
# Calculate statistics for base salary by borough
borough_stats = df.groupby('work_location_borough')['base_salary'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max'
]).round(2)

borough_stats = borough_stats.rename(columns={
    'count': 'Employee Count',
    'mean': 'Average Base Salary',
    'median': 'Median Base Salary',
    'std': 'Standard Deviation',
    'min': 'Minimum Salary',
    'max': 'Maximum Salary'
})

print("Base salary statistics by borough:")
print(borough_stats)

Base salary statistics by borough:
                       Employee Count  Average Base Salary  \
work_location_borough                                        
BRONX                             180             75951.56   
BROOKLYN                          180             77648.42   
MANHATTAN                         514             91199.38   
QUEENS                             95             81591.24   
RICHMOND                           31             78376.77   

                       Median Base Salary  Standard Deviation  Minimum Salary  \
work_location_borough                                                           
BRONX                             70106.0            22045.23          591.20   
BROOKLYN                          70106.0            24984.87        39206.00   
MANHATTAN                         78341.0            38675.69           37.67   
QUEENS                            70156.0            25231.11        48631.00   
RICHMOND                          70132.0   

In [53]:
# Sort by average base salary
print("\nSorted by average base salary (descending):")
print(borough_stats.sort_values('Average Base Salary', ascending=False))


Sorted by average base salary (descending):
                       Employee Count  Average Base Salary  \
work_location_borough                                        
MANHATTAN                         514             91199.38   
QUEENS                             95             81591.24   
RICHMOND                           31             78376.77   
BROOKLYN                          180             77648.42   
BRONX                             180             75951.56   

                       Median Base Salary  Standard Deviation  Minimum Salary  \
work_location_borough                                                           
MANHATTAN                         78341.0            38675.69           37.67   
QUEENS                            70156.0            25231.11        48631.00   
RICHMOND                          70132.0            19384.76        57127.00   
BROOKLYN                          70106.0            24984.87        39206.00   
BRONX                             

In [54]:
# Statistical testing - test if salary differences between boroughs are significant
# First, compare base salaries between Manhattan and other boroughs
manhattan_salaries = df[df['work_location_borough'] == 'MANHATTAN']['base_salary']
non_manhattan_salaries = df[df['work_location_borough'] != 'MANHATTAN']['base_salary']

print("Manhattan vs. Other Boroughs Salary Comparison:")
print(f"Manhattan average base salary: ${manhattan_salaries.mean():,.2f}")
print(f"Manhattan salary standard deviation: ${manhattan_salaries.std():,.2f}")
print(f"Manhattan employee count: {len(manhattan_salaries)}")
print(f"\nOther boroughs average base salary: ${non_manhattan_salaries.mean():,.2f}")
print(f"Other boroughs salary standard deviation: ${non_manhattan_salaries.std():,.2f}")
print(f"Other boroughs employee count: {len(non_manhattan_salaries)}")


Manhattan vs. Other Boroughs Salary Comparison:
Manhattan average base salary: $91,199.38
Manhattan salary standard deviation: $38,675.69
Manhattan employee count: 514

Other boroughs average base salary: $77,837.13
Other boroughs salary standard deviation: $23,674.81
Other boroughs employee count: 486


In [55]:
from scipy import stats

In [56]:
# Use t-test to check if the difference is significant
# Use independent samples t-test, assuming unequal variances (Welch's t-test)
t_stat, p_value = stats.ttest_ind(manhattan_salaries, non_manhattan_salaries, equal_var=False)
print(f"\nIndependent samples t-test results:")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("Conclusion: p-value < 0.05, the difference between Manhattan and other boroughs is statistically significant.")
else:
    print("Conclusion: p-value >= 0.05, the difference between Manhattan and other boroughs is not statistically significant.")


Independent samples t-test results:
t-statistic: 6.6288
p-value: 0.000000
Conclusion: p-value < 0.05, the difference between Manhattan and other boroughs is statistically significant.


In [57]:
# Calculate Manhattan salary premium relative to other boroughs
manhattan_mean = manhattan_salaries.mean()
non_manhattan_mean = non_manhattan_salaries.mean()
salary_premium = ((manhattan_mean - non_manhattan_mean) / non_manhattan_mean) * 100

print("Analysis Summary:")
print(f"1. Manhattan average base salary: ${manhattan_mean:,.2f}")
print(f"   Other boroughs average base salary: ${non_manhattan_mean:,.2f}")
print(f"2. Manhattan salary premium: {salary_premium:.2f}% higher than other boroughs")
print(f"3. Statistical test shows the difference is {'significant' if p_value < 0.05 else 'not significant'} (p = {p_value:.6f})")

Analysis Summary:
1. Manhattan average base salary: $91,199.38
   Other boroughs average base salary: $77,837.13
2. Manhattan salary premium: 17.17% higher than other boroughs
3. Statistical test shows the difference is significant (p = 0.000000)


In [58]:
print("\nHypothesis Testing:")
if p_value < 0.05 and manhattan_mean > non_manhattan_mean:
    print("✓ Accept the hypothesis: Employees in Manhattan have significantly higher average base salaries than those in other boroughs")
else:
    print("✗ Reject the hypothesis: Data does not support that Manhattan employees have significantly higher base salaries")


Hypothesis Testing:
✓ Accept the hypothesis: Employees in Manhattan have significantly higher average base salaries than those in other boroughs


In [63]:
# Statistical testing - Data Visualization

# Prepare the data needed
borough_means = df.groupby('work_location_borough')['base_salary'].mean().sort_values(ascending=False)
borough_means_df = borough_means.reset_index()
borough_means_df.columns = ['Borough', 'Average Base Salary']

# Prepare a bar chart
fig1 = px.bar(borough_means_df,
              x='Borough',
              y='Average Base Salary',
              title='Average Base Salary by Borough',
              color='Average Base Salary',
              color_continuous_scale='blues',
              text='Average Base Salary')

fig1.update_traces(texttemplate='$%{text:,.0f}',
                   textposition='outside',
                   marker_line_color='black',
                   marker_line_width=1)
fig1.update_layout(xaxis_title='Borough',
                   yaxis_title='Average Base Salary ($)',
                   xaxis_tickangle=-45,
                   coloraxis_showscale=False)

HTML(fig1.to_html(include_plotlyjs="cdn", full_html=False))


In [62]:
# Create a new column to distinguish between Manhattan and other boroughs
df['location_category'] = df['work_location_borough'].apply(
    lambda x: 'MANHATTAN' if x == 'MANHATTAN' else 'OTHER BOROUGHS'
)

# Box plot of base salary comparison: Manhattan vs other boroughs

fig2 = px.box(df,
              x='location_category',
              y='base_salary',
              title='Manhattan vs Other Boroughs: Base Salary Comparison',
              color='location_category',
              points="outliers")

fig2.update_layout(xaxis_title='Location Category',
                   yaxis_title='Base Salary ($)',
                   showlegend=False)

HTML(fig2.to_html(include_plotlyjs="cdn", full_html=False))